This notebook queries ATNF for a set of pulsars with their properties, filters this set of pulsars to those contained in the VASTGalactic footprint, and saves this set to a csv.
The notebook then queries the VAST observations to find detections of these pulsars, and saves these observations to another csv.

PRECONDITION: access to the VAST pipeline on this machine

IN: nothing

OUT: 'all_pulsar_measurements.csv', 'paper_dfv0.csv'

In [1]:
# Installing psrqpy to access known pulsar table
!pip install psrqpy

In [ ]:
# importing required modules
import psrqpy
from vasttools.query import Query
import pandas as pd
from vasttools.moc import VASTMOCS
from astropy import units as u
import numpy as np
from astropy.coordinates import SkyCoord

In [ ]:
# calculates average distance between detection in VAST and known source coordinates
def calc_avg_separation(source_df):
    sep_list = []
    for i in np.arange(source_df.shape[0]):
        if not (np.isnan(source_df['ra_deg_cont'][i]) or np.isnan(source_df['dec_deg_cont'][i])):
            detected_skycoord = SkyCoord(str(source_df['ra_deg_cont'][i]) + " " + str(source_df['dec_deg_cont'][i]), unit='deg')
            database_skycoord = source_df['skycoord'][i]
            sep_list.append(detected_skycoord.separation(database_skycoord).arcsec)
    return np.mean(sep_list)

In [ ]:
#computes how many times a source has been detected in VAST
def num_detections(source_df):
    bools = pd.isna(source_df['flux_peak'])
    counter = 0
    for item in bools:
        if not item:
            counter += 1
    return counter

In [ ]:
# Grab all the pulsars from ATNF catalog
pulsars = psrqpy.QueryATNF(params=["JNAME", "RAJD", "DECJD", "GB", "P0", "P1", "DM", "S1400", "ASSOC", "Age"]) 

In [ ]:
#loading survey MOC for filtering
vast_moc = VASTMOCS()
moc =  vast_moc.load_survey_footprint('full')

In [ ]:
# Give the RA and Dec values and get a mask (True if the pulsar is inside the field of view, false else)
in_pilot = moc.contains(pulsars.table['RAJD'].data*u.deg, pulsars.table['DECJD'].data*u.deg, keep_inside=True) 

filtered_psrs = pulsars.table[in_pilot] # Filter out pulsars not in VAST footprint

#throwing out psrs in globular clusters or with high error in RA and DEC (certainty less than about an arcsec) or zero error (weird)
final_atnf_table = filtered_psrs[(filtered_psrs['ASSOC']!="GC")
                                 &(filtered_psrs['DECJD_ERR']<0.001)
                                 &(filtered_psrs['RAJD_ERR']<0.001)
                                 &(filtered_psrs['DECJD_ERR']!=0)
                                 &(filtered_psrs['RAJD_ERR']!=0)]

final_df = final_atnf_table.to_pandas()

In [ ]:
#saving ATNF properties to a csv
final_df.to_csv('paper_dfv0.csv')

In [ ]:
#creating a list of the coordinates for each pulsar
my_coords = []
for i in np.arange(final_df.shape[0]):
    my_coords.append(str(final_df['RAJD'][i]) + " " + str(final_df['DECJD'][i]))
    
#creating a SkyCoord object out of the pulsar coordinates. This will be used to query the VAST data
my_skycoords = SkyCoord(my_coords, unit='deg')

In [ ]:
#querying the VAST data. Be sure to use the given kwargs
my_query = Query(coords=my_skycoords, epochs='all-vast', use_tiles=True, corrected_data=False)

#finding sources
my_query.find_sources()

In [ ]:
#getting rid of sources with less than three detections
labels = []
for i in np.arange(my_query.results.shape[0]):
    if (num_detections(my_query.results[i].measurements[my_query.results[i].measurements['freq']!=1367.5])<3):
        labels.append(my_query.results.index[i])
results = my_query.results.drop(labels) 

In [ ]:
#condensing all measurements into a single dataframe
measurements = []
names = []
for i in np.arange(results.shape[0]):
    data = results.iloc[i].measurements
    ra = data['ra'].iloc[0].value
    dec = data['dec'].iloc[0].value
    
    #finding pulsar that these measurements correspond to
    for i in np.arange(final_df.shape[0]):
        psr = final_df.iloc[i]
        if (psr['RAJD'].value==ra)&(psr['DECJD'].value==dec):
            name = psr['JNAME'].value
            break
            
    measurements.append(data)
    names.append(name)

psrs_measurements = pd.concat(measuremnts, keys=names)

In [ ]:
#saving that dataframe to a csv
psrs_measurements.to_csv('all_pulsar_measurements.csv')